In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../data/cars_clean.parquet")

df["age"] = (2023 - df["year"]).clip(lower=0)
df["miles_per_year"] = df["mileage"] / df["age"].clip(lower=1)

In [2]:
features = ["manufacturer", "model", "age", "mileage", "miles_per_year",
    "drivetrain", "fuel_type", "transmission_type",
    "accidents_or_damage", "one_owner", "personal_use_only",
    "mpg_city", "mpg_hwy", "driver_rating", "driver_reviews_num",
    "seller_rating", "seller_rating_missing"]

cat_cols = ["manufacturer", "model", "drivetrain", "fuel_type", "transmission_type",
            "accidents_or_damage", "one_owner", "personal_use_only"]

df[cat_cols] = df[cat_cols].astype("category")

In [3]:
train, test = train_test_split(df, test_size=0.2, random_state=42)
train_fit, valid = train_test_split(train, test_size=0.1, random_state=42)

In [4]:
import os, joblib
import lightgbm as lgb

os.makedirs("../models", exist_ok=True)

def train_lgbm(name, **extra):
    path = f"../models/{name}.joblib"
    if os.path.exists(path):
        print(f"Loaded {name}")
        return joblib.load(path)
    m = lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.05, num_leaves=127,
                          min_child_samples=20, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, random_state=42, verbose=-1, **extra)
    m.fit(train_fit[features], np.log(train_fit["price"]),
          eval_set=[(valid[features], np.log(valid["price"]))],
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)])
    joblib.dump(m, path)
    return m

mid  = train_lgbm("lgbm_mid")
low  = train_lgbm("lgbm_q10", objective="quantile", alpha=0.10)
high = train_lgbm("lgbm_q90", objective="quantile", alpha=0.90)

c:\Users\Manci\Desktop\GitHub projects\used-car-deal-detector\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[500]	valid_0's l2: 0.0139678
[1000]	valid_0's l2: 0.0136605
[1500]	valid_0's l2: 0.0135116
[2000]	valid_0's l2: 0.01344
[2500]	valid_0's l2: 0.0134164
Early stopping, best iteration is:
[2793]	valid_0's l2: 0.0134016


c:\Users\Manci\Desktop\GitHub projects\used-car-deal-detector\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[500]	valid_0's quantile: 0.0184965
[1000]	valid_0's quantile: 0.018353
[1500]	valid_0's quantile: 0.0183436
Early stopping, best iteration is:
[1457]	valid_0's quantile: 0.0183405


c:\Users\Manci\Desktop\GitHub projects\used-car-deal-detector\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[500]	valid_0's quantile: 0.0186998
[1000]	valid_0's quantile: 0.0185694
Early stopping, best iteration is:
[1063]	valid_0's quantile: 0.0185618


In [5]:
yv = np.log(valid["price"])
lo_v, hi_v = low.predict(valid[features]), high.predict(valid[features])
scores = np.maximum(lo_v - yv, yv - hi_v)
q = np.quantile(scores, 0.80)

In [6]:
X = test[features]
test = test.assign(
    pred_mid=np.exp(mid.predict(X)),
    pred_low=np.exp(low.predict(X) - q),
    pred_high=np.exp(high.predict(X) + q),
)
# Make sure low <= mid <= high
test["pred_low"] = test[["pred_low", "pred_mid"]].min(axis=1)
test["pred_high"] = test[["pred_high", "pred_mid"]].max(axis=1)

In [7]:
inside = test["price"].between(test["pred_low"], test["pred_high"]).mean()
below = (test["price"] < test["pred_low"]).mean()
above = (test["price"] > test["pred_high"]).mean()
print(f"Inside range: {inside:.1%}  (target ~80%)")
print(f"Below range:  {below:.1%}  (target ~10%)")
print(f"Above range:  {above:.1%}  (target ~10%)")

test["range_width_pct"] = (test["pred_high"] - test["pred_low"]) / test["pred_mid"]
bands = pd.cut(test["price"], [0, 10_000, 20_000, 35_000, 60_000, 200_000])
test.groupby(bands, observed=True)["range_width_pct"].median().round(3)

Inside range: 79.9%  (target ~80%)
Below range:  10.1%  (target ~10%)
Above range:  10.0%  (target ~10%)


price
(0, 10000]         0.488
(10000, 20000]     0.264
(20000, 35000]     0.196
(35000, 60000]     0.174
(60000, 200000]    0.171
Name: range_width_pct, dtype: float64

In [8]:
test["discount"] = 1 - test["price"] / test["pred_mid"]
deals = (test[test["price"] < test["pred_low"]]
         .sort_values("discount", ascending=False))

cols = ["manufacturer", "model", "year", "mileage", "accidents_or_damage",
        "price", "pred_low", "pred_mid", "pred_high", "discount"]
print(f"{len(deals):,} flagged of {len(test):,} test listings")
deals[cols].head(20).round(2)

15,137 flagged of 149,764 test listings


,manufacturer,model,year,mileage,accidents_or_damage,price,pred_low,pred_mid,pred_high,discount
82706,Cadillac,Series 62,1964,90000.0,No,6590.0,18739.23,53460.74,59325.62,0.88
210456,Ford,F-250 TOW,1976,88000.0,No,4990.0,12423.56,28495.08,58200.32,0.82
449753,Land Rover,LR4 HSE,2012,151377.0,Unknown,3000.0,10796.94,14152.47,17950.53,0.79
132382,Chevrolet,3100,1955,99999.0,Yes,11850.0,22601.44,51639.87,79718.67,0.77
75460,Buick,Verano Leather Group,2015,116510.0,Yes,2900.0,10642.00,11885.48,13272.92,0.76
388496,Jeep,CJ-7,1985,54985.0,No,9575.0,15078.65,39065.36,67663.39,0.75
83166,Cadillac,SRX Luxury Collection,2016,65321.0,Yes,4950.0,17268.16,19724.52,21236.04,0.75
209419,Ford,Excursion Limited,2001,212100.0,Unknown,3000.0,6609.37,10663.72,17557.06,0.72
481660,Lincoln,MKT EcoBoost,2014,134.0,Yes,9995.0,20251.35,33424.03,52796.56,0.70
328803,Honda,Accord Hybrid Base,2022,1132.0,No,9999.0,29979.23,33355.43,34040.40,0.70


In [9]:
df.loc[df["price"] == 2500, "seller_name"].value_counts().head(10)

seller_name
Car Collection Inc                   7
Suburban Ford of Sterling Heights    2
Corwin Public Wholesale              2
REM Motors LLC                       2
Central FL Motors LLC                2
Shea Buick GMC                       1
Mike Molstead Motors                 1
Starfire Auto Inc.                   1
Putnam Auto Sales Inc                1
A2Z Auto Group                       1
Name: count, dtype: int64

In [10]:
def classify(row):
    if row["year"] < 2000:
        return "Classic (out of scope)"
    if row["discount"] > 0.5:
        return "Likely data error"
    if row["accidents_or_damage"] == "Yes":
        return "Accident history"
    return "Possible deal"

deals["flag_type"] = deals.apply(classify, axis=1)
deals["flag_type"].value_counts()

flag_type
Possible deal             11230
Accident history           3651
Classic (out of scope)      131
Likely data error           125
Name: count, dtype: int64

In [ ]:
test["discount"] = 1 - test["price"] / test["pred_mid"]
deals = (test[test["price"] < test["pred_low"]]
         .sort_values("discount", ascending=False))

cols = ["manufacturer", "model", "year", "mileage", "accidents_or_damage",
        "price", "pred_low", "pred_mid", "pred_high", "discount"]
print(f"{len(deals):,} flagged of {len(test):,} test listings")
deals[cols].head(20).round(2)

15,137 flagged of 149,764 test listings


,manufacturer,model,year,mileage,accidents_or_damage,price,pred_low,pred_mid,pred_high,discount
82706,Cadillac,Series 62,1964,90000.0,No,6590.0,18739.23,53460.74,59325.62,0.88
210456,Ford,F-250 TOW,1976,88000.0,No,4990.0,12423.56,28495.08,58200.32,0.82
449753,Land Rover,LR4 HSE,2012,151377.0,Unknown,3000.0,10796.94,14152.47,17950.53,0.79
132382,Chevrolet,3100,1955,99999.0,Yes,11850.0,22601.44,51639.87,79718.67,0.77
75460,Buick,Verano Leather Group,2015,116510.0,Yes,2900.0,10642.00,11885.48,13272.92,0.76
388496,Jeep,CJ-7,1985,54985.0,No,9575.0,15078.65,39065.36,67663.39,0.75
83166,Cadillac,SRX Luxury Collection,2016,65321.0,Yes,4950.0,17268.16,19724.52,21236.04,0.75
209419,Ford,Excursion Limited,2001,212100.0,Unknown,3000.0,6609.37,10663.72,17557.06,0.72
481660,Lincoln,MKT EcoBoost,2014,134.0,Yes,9995.0,20251.35,33424.03,52796.56,0.70
328803,Honda,Accord Hybrid Base,2022,1132.0,No,9999.0,29979.23,33355.43,34040.40,0.70


In [ ]:
test["discount"] = 1 - test["price"] / test["pred_mid"]
deals = (test[test["price"] < test["pred_low"]]
         .sort_values("discount", ascending=False))

cols = ["manufacturer", "model", "year", "mileage", "accidents_or_damage",
        "price", "pred_low", "pred_mid", "pred_high", "discount"]
print(f"{len(deals):,} flagged of {len(test):,} test listings")
deals[cols].head(20).round(2)

15,137 flagged of 149,764 test listings


,manufacturer,model,year,mileage,accidents_or_damage,price,pred_low,pred_mid,pred_high,discount
82706,Cadillac,Series 62,1964,90000.0,No,6590.0,18739.23,53460.74,59325.62,0.88
210456,Ford,F-250 TOW,1976,88000.0,No,4990.0,12423.56,28495.08,58200.32,0.82
449753,Land Rover,LR4 HSE,2012,151377.0,Unknown,3000.0,10796.94,14152.47,17950.53,0.79
132382,Chevrolet,3100,1955,99999.0,Yes,11850.0,22601.44,51639.87,79718.67,0.77
75460,Buick,Verano Leather Group,2015,116510.0,Yes,2900.0,10642.00,11885.48,13272.92,0.76
388496,Jeep,CJ-7,1985,54985.0,No,9575.0,15078.65,39065.36,67663.39,0.75
83166,Cadillac,SRX Luxury Collection,2016,65321.0,Yes,4950.0,17268.16,19724.52,21236.04,0.75
209419,Ford,Excursion Limited,2001,212100.0,Unknown,3000.0,6609.37,10663.72,17557.06,0.72
481660,Lincoln,MKT EcoBoost,2014,134.0,Yes,9995.0,20251.35,33424.03,52796.56,0.70
328803,Honda,Accord Hybrid Base,2022,1132.0,No,9999.0,29979.23,33355.43,34040.40,0.70


In [ ]:
test["discount"] = 1 - test["price"] / test["pred_mid"]
deals = (test[test["price"] < test["pred_low"]]
         .sort_values("discount", ascending=False))

cols = ["manufacturer", "model", "year", "mileage", "accidents_or_damage",
        "price", "pred_low", "pred_mid", "pred_high", "discount"]
print(f"{len(deals):,} flagged of {len(test):,} test listings")
deals[cols].head(20).round(2)

15,137 flagged of 149,764 test listings


,manufacturer,model,year,mileage,accidents_or_damage,price,pred_low,pred_mid,pred_high,discount
82706,Cadillac,Series 62,1964,90000.0,No,6590.0,18739.23,53460.74,59325.62,0.88
210456,Ford,F-250 TOW,1976,88000.0,No,4990.0,12423.56,28495.08,58200.32,0.82
449753,Land Rover,LR4 HSE,2012,151377.0,Unknown,3000.0,10796.94,14152.47,17950.53,0.79
132382,Chevrolet,3100,1955,99999.0,Yes,11850.0,22601.44,51639.87,79718.67,0.77
75460,Buick,Verano Leather Group,2015,116510.0,Yes,2900.0,10642.00,11885.48,13272.92,0.76
388496,Jeep,CJ-7,1985,54985.0,No,9575.0,15078.65,39065.36,67663.39,0.75
83166,Cadillac,SRX Luxury Collection,2016,65321.0,Yes,4950.0,17268.16,19724.52,21236.04,0.75
209419,Ford,Excursion Limited,2001,212100.0,Unknown,3000.0,6609.37,10663.72,17557.06,0.72
481660,Lincoln,MKT EcoBoost,2014,134.0,Yes,9995.0,20251.35,33424.03,52796.56,0.70
328803,Honda,Accord Hybrid Base,2022,1132.0,No,9999.0,29979.23,33355.43,34040.40,0.70


In [ ]:
test["discount"] = 1 - test["price"] / test["pred_mid"]
deals = (test[test["price"] < test["pred_low"]]
         .sort_values("discount", ascending=False))

cols = ["manufacturer", "model", "year", "mileage", "accidents_or_damage",
        "price", "pred_low", "pred_mid", "pred_high", "discount"]
print(f"{len(deals):,} flagged of {len(test):,} test listings")
deals[cols].head(20).round(2)

15,137 flagged of 149,764 test listings


,manufacturer,model,year,mileage,accidents_or_damage,price,pred_low,pred_mid,pred_high,discount
82706,Cadillac,Series 62,1964,90000.0,No,6590.0,18739.23,53460.74,59325.62,0.88
210456,Ford,F-250 TOW,1976,88000.0,No,4990.0,12423.56,28495.08,58200.32,0.82
449753,Land Rover,LR4 HSE,2012,151377.0,Unknown,3000.0,10796.94,14152.47,17950.53,0.79
132382,Chevrolet,3100,1955,99999.0,Yes,11850.0,22601.44,51639.87,79718.67,0.77
75460,Buick,Verano Leather Group,2015,116510.0,Yes,2900.0,10642.00,11885.48,13272.92,0.76
388496,Jeep,CJ-7,1985,54985.0,No,9575.0,15078.65,39065.36,67663.39,0.75
83166,Cadillac,SRX Luxury Collection,2016,65321.0,Yes,4950.0,17268.16,19724.52,21236.04,0.75
209419,Ford,Excursion Limited,2001,212100.0,Unknown,3000.0,6609.37,10663.72,17557.06,0.72
481660,Lincoln,MKT EcoBoost,2014,134.0,Yes,9995.0,20251.35,33424.03,52796.56,0.70
328803,Honda,Accord Hybrid Base,2022,1132.0,No,9999.0,29979.23,33355.43,34040.40,0.70


In [ ]:
test["discount"] = 1 - test["price"] / test["pred_mid"]
deals = (test[test["price"] < test["pred_low"]]
         .sort_values("discount", ascending=False))

cols = ["manufacturer", "model", "year", "mileage", "accidents_or_damage",
        "price", "pred_low", "pred_mid", "pred_high", "discount"]
print(f"{len(deals):,} flagged of {len(test):,} test listings")
deals[cols].head(20).round(2)

15,137 flagged of 149,764 test listings


,manufacturer,model,year,mileage,accidents_or_damage,price,pred_low,pred_mid,pred_high,discount
82706,Cadillac,Series 62,1964,90000.0,No,6590.0,18739.23,53460.74,59325.62,0.88
210456,Ford,F-250 TOW,1976,88000.0,No,4990.0,12423.56,28495.08,58200.32,0.82
449753,Land Rover,LR4 HSE,2012,151377.0,Unknown,3000.0,10796.94,14152.47,17950.53,0.79
132382,Chevrolet,3100,1955,99999.0,Yes,11850.0,22601.44,51639.87,79718.67,0.77
75460,Buick,Verano Leather Group,2015,116510.0,Yes,2900.0,10642.00,11885.48,13272.92,0.76
388496,Jeep,CJ-7,1985,54985.0,No,9575.0,15078.65,39065.36,67663.39,0.75
83166,Cadillac,SRX Luxury Collection,2016,65321.0,Yes,4950.0,17268.16,19724.52,21236.04,0.75
209419,Ford,Excursion Limited,2001,212100.0,Unknown,3000.0,6609.37,10663.72,17557.06,0.72
481660,Lincoln,MKT EcoBoost,2014,134.0,Yes,9995.0,20251.35,33424.03,52796.56,0.70
328803,Honda,Accord Hybrid Base,2022,1132.0,No,9999.0,29979.23,33355.43,34040.40,0.70


In [11]:
real = deals[deals["flag_type"] == "Possible deal"]
real[cols].head(20).round(2)

,manufacturer,model,year,mileage,accidents_or_damage,price,pred_low,pred_mid,pred_high,discount
714621,Toyota,Yaris iA STD,2017,45571.0,No,15939.0,22598.59,31871.71,44066.24,0.50
167565,Dodge,Ram 1500 SLT,2008,201500.0,No,4290.0,6350.70,8576.51,12301.25,0.50
530517,Mercedes-Benz,M-Class ML 350 4MATIC,2005,157238.0,No,3991.0,5371.89,7978.47,11492.99,0.50
367323,Jaguar,XF Luxury,2009,166000.0,No,2995.0,4041.26,5981.18,10220.94,0.50
161681,Chrysler,PT Cruiser Touring,2009,129000.0,Unknown,2400.0,3609.43,4779.57,7804.33,0.50
164350,Dodge,Grand Caravan SXT,2010,243207.0,No,2900.0,3623.29,5774.03,9543.69,0.50
709886,Toyota,Tundra SR5,2002,219000.0,No,4900.0,6438.18,9746.26,13115.35,0.50
187893,Ford,Focus Titanium,2014,60000.0,Unknown,6900.0,11064.18,13715.70,16998.76,0.50
510900,Mercedes-Benz,"Sprinter 2500 2500 Standard Roof V6 144"" 4WD",2022,921.0,No,60888.0,64979.07,120624.33,198672.79,0.50
544920,Mitsubishi,Eclipse GT,2006,166265.0,Unknown,3150.0,3498.16,6198.68,9095.77,0.49
